In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import datetime
import uuid

fake = Faker()
np.random.seed(42)

print("Audit-Grade Registry for Romina PLC")

Audit-Grade Registry for Romina PLC


In [2]:
ROMINA_NETWORK = [
    {"Station": "Astatke Menafesha", "Region": "Sidama", "Woreda": "Wonsho", "Specialty": True},
    {"Station": "Belete Tanga", "Region": "Sidama", "Woreda": "Bona", "Specialty": True},
    {"Station": "Zenebe Dikicha", "Region": "Sidama", "Woreda": "Aroresa", "Specialty": True},
    {"Station": "Dogada Yemasira", "Region": "Oromia", "Woreda": "Nansabo", "Specialty": True},
    {"Station": "Michael Girma", "Region": "Oromia", "Woreda": "Anfilo", "Specialty": True},
    {"Station": "Hagire Buticha", "Region": "Oromia", "Woreda": "Nansabo", "Specialty": False},
    {"Station": "Oddi Boku", "Region": "Sidama", "Woreda": "Girja", "Specialty": False},
    {"Station": "Abebe Jengelo", "Region": "Sidama", "Woreda": "Hoko Girja", "Specialty": False}
]

DEFECTS = ["High Moisture (>12.5%)", "Primary Defects", "Under-dried", "Over-fermented", "None"]

In [3]:
def generate_romina_audit_data(records=400):
    data = []
    for _ in range(records):
        st = np.random.choice(ROMINA_NETWORK)
        
        # Date: Oct 2024 to Dec 2025 (Full Harvest Cycle)
        insp_date = fake.date_between(start_date=datetime.date(2024, 10, 1), end_date=datetime.date(2025, 12, 31))
        
        # Status Logic: 22% Rejection (Maintaining your project's problem scale)
        is_rejected = np.random.random() < 0.22
        status = "Rejected" if is_rejected else "Passed"
        defect = np.random.choice(DEFECTS[:-1]) if is_rejected else "None"
        
        # Cup Scoring (SCA Standards)
        score_base = 83 if st["Specialty"] else 79
        cup_score = round(score_base + np.random.uniform(0, 7), 2) if status == "Passed" else round(np.random.uniform(72, 79), 2)

        data.append({
            "Lot_ID": f"ROM-25-{uuid.uuid4().hex[:5].upper()}",
            "Station": st["Station"],
            "Woreda": st["Woreda"],
            "Region": st["Region"],
            "Inspection_Date": insp_date,
            "Quantity_MT": round(np.random.uniform(18.5, 19.5), 2), # 1 Full Container (FCL)
            "Status": status,
            "Defect_Type": defect,
            "Cup_Score": cup_score,
            "EUDR_Verified": np.random.choice([True, False], p=[0.8, 0.2])
        })
    return pd.DataFrame(data)

df_registry = generate_romina_audit_data()

In [5]:
# Validation: Ensure the scale makes sense for a private PLC
total_mt = df_registry['Quantity_MT'].sum()
rejections = len(df_registry[df_registry['Status'] == 'Rejected'])

print(f"--- REGISTRY AUDIT COMPLETE ---")
print(f"Total Records: {len(df_registry)}")
print(f"Total Volume: {total_mt:,.2f} MT ")
print(f"Total Quality Rejections: {rejections} Lots")

df_registry.to_csv('../data/raw/romina_registry_v2.csv', index=False)

--- REGISTRY AUDIT COMPLETE ---
Total Records: 400
Total Volume: 7,583.35 MT 
Total Quality Rejections: 93 Lots


In [6]:
df_registry.head(5)

,Lot_ID,Station,Woreda,Region,Inspection_Date,Quantity_MT,Status,Defect_Type,Cup_Score,EUDR_Verified
0,ROM-25-645E5,Oddi Boku,Girja,Sidama,2025-10-12,19.28,Passed,None,80.28,True
1,ROM-25-F9562,Belete Tanga,Bona,Sidama,2024-10-10,18.83,Rejected,Under-dried,75.21,True
2,ROM-25-85A32,Zenebe Dikicha,Aroresa,Sidama,2025-02-27,19.44,Rejected,Primary Defects,77.05,True
3,ROM-25-2F713,Dogada Yemasira,Nansabo,Oromia,2025-06-23,18.51,Rejected,Over-fermented,76.28,True
4,ROM-25-63574,Zenebe Dikicha,Aroresa,Sidama,2025-07-08,18.79,Passed,None,83.98,True
